# Plate ID through time for a single (lon, lat) point

This notebook reports the **plate ID** assigned by a topological plate model to a present-day (lon, lat) point at each **1-My** time step between a user-supplied start and end time.

It uses the same plate-ID assignment logic as `carbonate_sediment_thickness_2026.ipynb` &mdash; namely, `pygplates.resolve_topologies()` followed by a point-in-polygon test against the resolved-topology polygons.

## Workflow

1. Load the rotation and topology files. You can choose between:
   - **GPlately plate-model-manager** (downloads/caches a named model, e.g. `Alfonso2024`), or
   - **A custom plate model** that you point to with explicit local rotation/topology file paths. The default below is configured for the **Heidari_V25** model in this folder.
2. Resolve topologies at present day and find the plate ID `P0` of the polygon that contains the input lon/lat. This is the carrier plate.
3. For each time `t` in 1-My steps from `start_time` to `end_time`:
    - Reconstruct the input point back to time `t` using `P0` (the point moves with its present-day plate).
    - Resolve topologies at time `t`, look up the plate ID `Pt` of the polygon containing the reconstructed paleo location.
    - **Print this row immediately to the screen** so you can watch progress live.
4. After the loop finishes, save the full table to a CSV.

## Output columns

| Column | Meaning |
|---|---|
| `time_Ma` | Paleo time (Ma) |
| `input_lon`, `input_lat` | The user-supplied present-day point |
| `reconstructed_lon`, `reconstructed_lat` | Paleo location of the input point at time `t`, reconstructed using the carrier plate ID `P0` |
| `carrier_plate_id` | Present-day plate ID `P0` (constant for all rows) |
| `plate_id_at_time` | Plate ID `Pt` of the resolved-topology polygon at the reconstructed paleo location at time `t` (this is what the topological model assigns to that paleo location at that time) |


## 1. Imports

In [ ]:
import csv
import glob
import os
import sys

import pygplates

try:
    import pandas as pd
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False

## 2. User inputs

Edit the values in this cell to control the run.

* `lon`, `lat` &mdash; the **present-day** location of interest (degrees).
* `start_time`, `end_time` &mdash; the time range (Ma). Step is fixed at 1 My.
  `start_time` may be larger or smaller than `end_time`; iteration is inclusive at both ends.
* `anchor_plate_id` &mdash; the fixed reference-frame plate ID used by `pygplates.resolve_topologies` and for the reconstruction. For the Heidari_V25 model this is set to `705705`.
* `output_csv` &mdash; where to save the results.

### Choosing the plate model

Set `using_local_model = False` to use the **GPlately plate-model-manager** (downloads/caches a named model). Set `using_local_model = True` to use a **custom plate model** by listing rotation and topology files explicitly. The defaults below point at the local **Heidari_V25** model.

In [ ]:
# --- Point of interest (present day) ---------------------------------------
lon = 80.0       # degrees, in [-180, 180]
lat = -10.0      # degrees, in [-90, 90]

# --- Time range (Ma), 1-My step --------------------------------------------
start_time = 0
end_time   = 45

# --- Reference frame -------------------------------------------------------
# Fixed anchor plate ID for the Heidari_V25 model.
anchor_plate_id = 705705

# --- Output ----------------------------------------------------------------
output_csv = f"plate_id_through_time_{lon}_{lat}_{int(start_time)}_{int(end_time)}.csv"

# --- Plate model choice ----------------------------------------------------
# True  -> use a custom plate model (set the patterns below)
# False -> use the GPlately plate-model-manager (set `plate_model_name` below)
using_local_model = True

# Used only when using_local_model == False
plate_model_name = "Zahirovic2022"
plate_model_dir  = "plate-model-repo"   # local cache directory for downloaded files

# Used only when using_local_model == True --------------------------------
# Configured for the Heidari_V25 model living next to this notebook.
#
# Rotation file lives in   Heidari_V25/Rotation/CombinedRotation.rot
# Topology features live in:
#     Heidari_V25/Features/                 (topological points/lines, regional features)
#     Heidari_V25/Plate Boundaries/         (Plate_Boundaries.gpml)
#     Heidari_V25/Deformation Network/      (deforming-network meshes and active/inactive SZs)
#
# Patterns are passed to glob.glob() with no recursion. Adjust if you move/rename files.
local_rotation_patterns = [
    os.path.join("Heidari_V25", "Rotation", "*.rot"),
]
local_topology_patterns = [
    os.path.join("Heidari_V25", "Features", "*.gpml"),
    os.path.join("Heidari_V25", "Plate Boundaries", "*.gpml"),
    os.path.join("Heidari_V25", "Deformation Network", "*.gpml"),
]

## 3. Load rotation and topology files

If `using_local_model == False`, the GPlately plate-model-manager downloads (or reuses a cache of) the named model into `plate_model_dir`.

If `using_local_model == True`, the glob patterns above are expanded and the resulting file lists are used directly. Duplicates (same file resolved by multiple patterns) are removed.

In [ ]:
if using_local_model:
    rotation_filenames = []
    for pattern in local_rotation_patterns:
        matched = glob.glob(pattern)
        rotation_filenames.extend(matched if matched else [])

    topology_filenames = []
    for pattern in local_topology_patterns:
        matched = glob.glob(pattern)
        topology_filenames.extend(matched if matched else [])

    # De-duplicate while preserving order.
    rotation_filenames = list(dict.fromkeys(rotation_filenames))
    topology_filenames = list(dict.fromkeys(topology_filenames))

    if not rotation_filenames:
        raise FileNotFoundError(
            f"No rotation files matched any of the patterns: {local_rotation_patterns}\n"
            f"  (current working directory: {os.getcwd()})"
        )
    if not topology_filenames:
        raise FileNotFoundError(
            f"No topology files matched any of the patterns: {local_topology_patterns}\n"
            f"  (current working directory: {os.getcwd()})"
        )

    print("Using custom plate model:")
    print(f"  rotation files ({len(rotation_filenames)}):")
    for f in rotation_filenames:
        print(f"    {f}")
    print(f"  topology files ({len(topology_filenames)}):")
    for f in topology_filenames:
        print(f"    {f}")

else:
    from plate_model_manager import PlateModelManager
    pmm = PlateModelManager()
    plate_model = pmm.get_model(plate_model_name, data_dir=plate_model_dir)
    rotation_filenames = plate_model.get_rotation_model()
    topology_filenames = plate_model.get_topologies()
    print(f"Using plate-model-manager model: {plate_model_name!r} (cached in {plate_model_dir!r})")
    print(f"  {len(rotation_filenames)} rotation file(s), {len(topology_filenames)} topology file(s)")

rotation_model = pygplates.RotationModel(rotation_filenames)

## 4. Helper: plate ID at a point and time

Given a `pygplates.PointOnSphere`, resolve topologies at `time` (using `anchor_plate_id` as the reference frame) and return the plate ID of the resolved-topology polygon containing the point. Returns `0` if the point falls outside all topologies (this is the same fallback used in `carbonate_sediment_thickness.py`).

In [ ]:
def plate_id_at_point(point, time, anchor_plate_id=anchor_plate_id):
    resolved_topologies = []
    pygplates.resolve_topologies(
        topology_filenames,
        rotation_model,
        resolved_topologies,
        time,
        anchor_plate_id=anchor_plate_id,
    )
    for resolved_topology in resolved_topologies:
        boundary = resolved_topology.get_resolved_boundary()
        if boundary.is_point_in_polygon(point):
            return resolved_topology.get_feature().get_reconstruction_plate_id()
    return 0

## 5. Determine the present-day (carrier) plate ID `P0`

In [ ]:
# Note: pygplates.PointOnSphere takes (lat, lon) ordering, matching the parent notebook.
present_point = pygplates.PointOnSphere(lat, lon)

carrier_plate_id = plate_id_at_point(present_point, time=0, anchor_plate_id=anchor_plate_id)
print(f"Present-day plate ID at (lon={lon}, lat={lat}): {carrier_plate_id}")

## 6. Iterate through time &mdash; live output

For each time `t`:

1. Build the rotation that takes the present-day point on `carrier_plate_id` back to its paleo position at time `t` (in the chosen anchor-plate reference frame).
2. Apply that rotation to the input point to get its reconstructed (paleo) lon/lat.
3. Look up the plate ID of the resolved topology polygon at the reconstructed paleo location at time `t`.
4. **Print one row to the screen as it is computed** &mdash; you can watch the plate ID evolve as a function of time while the loop runs.

If the point straddles a plate boundary that has changed through time, `plate_id_at_time` may differ from `carrier_plate_id` &mdash; that is the topological model's view of which plate is at that paleo location.

In [ ]:
# Build inclusive 1-My step list (works whether start <= end or start > end).
if end_time >= start_time:
    times = [start_time + i for i in range(int(end_time - start_time) + 1)]
else:
    times = [start_time - i for i in range(int(start_time - end_time) + 1)]

# Live header (printed once). Each time step prints a row immediately after it's computed.
print(f"{'time_Ma':>8}  {'recon_lon':>11}  {'recon_lat':>10}  {'carrier_pid':>11}  {'plate_id_at_time':>16}")
print("-" * 66)
sys.stdout.flush()

rows = []
for t in times:
    if t == 0:
        recon_lat, recon_lon = lat, lon
        plate_id_t = carrier_plate_id
    else:
        # Rotation taking the present-day point on `carrier_plate_id` back to time t.
        stage_rotation = rotation_model.get_rotation(
            t, carrier_plate_id, 0, anchor_plate_id=anchor_plate_id,
        )
        paleo_point = stage_rotation * present_point
        recon_lat, recon_lon = paleo_point.to_lat_lon()

        # Plate ID at the reconstructed paleo location at time t.
        plate_id_t = plate_id_at_point(
            pygplates.PointOnSphere(recon_lat, recon_lon),
            time=t,
            anchor_plate_id=anchor_plate_id,
        )

    rows.append({
        "time_Ma": t,
        "input_lon": lon,
        "input_lat": lat,
        "reconstructed_lon": recon_lon,
        "reconstructed_lat": recon_lat,
        "carrier_plate_id": carrier_plate_id,
        "plate_id_at_time": plate_id_t,
    })

    # Live print this row to the screen as it's computed.
    print(f"{t:>8}  {recon_lon:>11.4f}  {recon_lat:>10.4f}  {carrier_plate_id:>11}  {plate_id_t:>16}")
    sys.stdout.flush()

print("-" * 66)
print(f"Computed plate IDs at {len(rows)} time(s).")

## 7. Display the results as a table

(Optional &mdash; the live loop above already printed the values as they were computed. This cell shows the same data as a tidy table.)

In [ ]:
if HAVE_PANDAS:
    df = pd.DataFrame(rows)
    display(df)
else:
    header = list(rows[0].keys())
    print("\t".join(header))
    for row in rows:
        print("\t".join(str(row[h]) for h in header))

## 8. Save to CSV

In [ ]:
out_dir = os.path.dirname(output_csv)
if out_dir and not os.path.exists(out_dir):
    os.makedirs(out_dir)

header = list(rows[0].keys())
with open(output_csv, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=header)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to: {os.path.abspath(output_csv)}")

## 9. (Optional) Quick visualisation

A simple step plot of `plate_id_at_time` vs. `time_Ma`. Re-run after editing the inputs above to compare different points.

In [ ]:
try:
    import matplotlib.pyplot as plt

    times_ma   = [r["time_Ma"] for r in rows]
    plate_ids  = [r["plate_id_at_time"] for r in rows]

    fig, ax = plt.subplots(figsize=(9, 3))
    ax.step(times_ma, plate_ids, where="post")
    ax.set_xlabel("Time (Ma)")
    ax.set_ylabel("Plate ID at reconstructed location")
    ax.set_title(f"Plate ID through time at (lon={lon}, lat={lat})  |  carrier plate ID = {carrier_plate_id}")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed; skipping plot.")